In [ ]:
from pathlib import Path
from itertools import combinations

import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_curve, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score


def calculate_probability(enrichment_scores: pd.DataFrame, cluster_names: list, predicted_cluster: str, omics_names: list, omics_weights: list) -> pd.DataFrame:
    """Calculate cluster prediction probability across omics combinations.

    Args:
        enrichment_scores (pd.DataFrame):
        cluster_names (list): Cluster names.
        predicted_cluster (str): Predicted cluster.
        omics_names (list): Omics names.
        omics_weights (list): Weight for omics.

    Returns:
        pd.DataFrame: Predicted probability.

    """
    other_clusters = [i for i in cluster_names if i != predicted_cluster]
    weights = {k: v for k, v in zip(omics_names, omics_weights)}

    # probability of each omics
    cluster_probability = pd.DataFrame(index=enrichment_scores.index.tolist(), columns=omics_names)
    for omics, weight in zip(omics_names, omics_weights):
        denominator = enrichment_scores[['%s_%s' % (omics, cluster) for cluster in other_clusters]].max(axis=1)
        numerator = enrichment_scores['%s_%s' % (omics, predicted_cluster)]
        probability = numerator / denominator
        cluster_probability[omics] = probability.loc[cluster_probability.index]

    omics_groups = []
    for i in range(1, len(omics_names) + 1):
        omics_groups.extend(combinations(omics_names, i))
    for group in omics_groups:
        group_name = 'combination_%s' % ('_'.join(group))
        cluster_probability[group_name] = 0
        for name in group:
            cluster_probability[group_name] = cluster_probability[group_name] + weights[name] * cluster_probability[name]

    return cluster_probability


def find_best_thresholds(probability: pd.DataFrame, y_true: list) -> dict:
    """Find best threshold based on F1 score.

    Args:
        probability (pd.DataFrame): Predicted probability.
        y_true (list): Ground truth.

    Returns:
        dict: Best thresholds.

    """
    best_thresholds = {}
    for combination in probability.columns[probability.columns.str.startswith('combination_')]:
        y_prob = probability[combination].tolist()
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
        best_threshold = thresholds[np.argmax(f1[:-1])]
        best_thresholds[combination] = best_threshold.item()

    return best_thresholds


def calculate_metrics(test_preds: pd.DataFrame, test_labels: list) -> pd.DataFrame:
    """Calculate evaluation metrics.

    Args:
        test_preds (pd.DataFrame): Cluster predictions across different omics combinations.
        test_labels (list):  Ground truth.

    Returns:
        pd.DataFrame: Evaluation metrics.

    """
    metrics = {}
    best_thresholds = {}
    for combination in test_preds.columns:
        y_preds = test_preds[combination].tolist()
        metrics[combination] = {}
        metrics[combination]['accuracy'] = accuracy_score(test_labels, y_preds)
        metrics[combination]['balanced_accuracy_score'] = balanced_accuracy_score(test_labels, y_preds)
        metrics[combination]['precision'] = precision_score(test_labels, y_preds)
        metrics[combination]['recall'] = recall_score(test_labels, y_preds)
        metrics[combination]['f1'] = f1_score(test_labels, y_preds)
    metrics = pd.DataFrame.from_dict(metrics, orient='index')

    return metrics

In [41]:
#### Select Best Number of Markers for Each Omics ####

enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/'
sample_info_file = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/data/sample_info.csv'
save_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/select_best_n_markers/'

predicted_cluster = 'nmf3'
omics_names = ['pro', 'rna', 'phospho']
omics_weights = [1, 1, 1]

f1 = {k: {} for k in omics_names}

for d in Path(enrichment_scores_dir).glob('*/'):
    # load data
    enrichment_scores = None
    cluster_names = []
    for f in Path(str(d)).glob('*/*.csv'):
        cluster = f.stem
        omics = f.parent.stem
        cluster_names.append(cluster)
        df = pd.read_csv(str(f), sep=',', index_col=0, header=0)
        if enrichment_scores is None:
            enrichment_scores = df['NES'].to_frame()
            enrichment_scores.columns = ['%s_%s' % (omics, cluster)]
        else:
            sample_names = df.index.intersection(enrichment_scores.index).tolist()
            enrichment_scores = enrichment_scores.loc[sample_names]
            enrichment_scores['%s_%s' % (omics, cluster)] = df.loc[sample_names, 'NES']
    cluster_names = list(set(cluster_names))
    sample_info = pd.read_csv(sample_info_file, sep=',', index_col=0, header=0)
    sample_info.index = sample_info['Tumor_Sample_Barcode'].tolist()
    sample_info['cluster'] = ['nmf%.1d' % cluster for cluster in sample_info['NMFcluster']]

    # calculate probability
    probability = calculate_probability(
        enrichment_scores,
        cluster_names=cluster_names, predicted_cluster=predicted_cluster,
        omics_names=omics_names, omics_weights=omics_weights
    )

    # leave-one-out
    test_preds = {}
    test_labels = []
    for test_index in probability.index:
        train_index = probability.index[probability.index != test_index].tolist()
        train_probability = probability.loc[train_index]
        train_y_true = (sample_info.loc[train_index, 'cluster'] == predicted_cluster).tolist()
        best_thresholds = find_best_thresholds(train_probability, train_y_true)
        test_probability = probability.loc[test_index]
        test_y_true = sample_info.loc[test_index, 'cluster'] == predicted_cluster
        for k, v in best_thresholds.items():
            if k not in test_preds:
                test_preds[k] = []
            test_preds[k].append(test_probability.loc[k].item() > best_thresholds[k])
        test_labels.append(test_y_true)
    test_preds = pd.DataFrame.from_dict(test_preds, orient='columns')
    metrics = calculate_metrics(test_preds, test_labels)

    for omics_name in omics_names:
        f1[omics_name][d.stem] = float(metrics.loc['combination_%s' % omics_name, 'f1'])

f1 = pd.DataFrame.from_dict(f1, orient='index')
f1['best'] = f1.idxmax(axis=1)

Path(save_dir).mkdir(parents=True, exist_ok=True)
f1.to_csv('%s/%s_f1.csv' % (save_dir, predicted_cluster), sep=',', index=True, header=True)
f1
# nmf1 - pro: 50, rna: 400, phospho: 700
# nmf2 - pro: 1100, rna: 1700, phospho: 50
# nmf3 - pro: 100, rna: 400, phospho: 1700

,n_markers_1700,n_markers_1100,n_markers_900,n_markers_1500,n_markers_600,n_markers_1300,n_markers_1400,n_markers_2000,n_markers_500,n_markers_800,...,n_markers_1200,n_markers_1000,n_markers_700,n_markers_50,n_markers_400,n_markers_1600,n_markers_200,n_markers_300,n_markers_1900,best
pro,0.529412,0.645161,0.557377,0.542857,0.709677,0.700000,0.606061,0.463768,0.709677,0.698413,...,0.688525,0.580645,0.566667,0.774194,0.642857,0.537313,0.700000,0.745763,0.500000,n_markers_100
rna,0.666667,0.698413,0.698413,0.666667,0.698413,0.666667,0.666667,0.666667,0.776119,0.698413,...,0.698413,0.698413,0.698413,0.793651,0.794118,0.666667,0.716418,0.716418,0.666667,n_markers_400
phospho,0.479339,0.479339,0.479339,0.479339,0.479339,0.479339,0.479339,0.479339,0.479339,0.479339,...,0.479339,0.479339,0.479339,0.479339,0.479339,0.479339,0.466667,0.466667,0.479339,n_markers_1700


In [42]:
# nmf1 - pro: 50, rna: 400, phospho: 700
pro_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_50/pro/'
rna_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_400/rna/'
phospho_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_700/phospho/'

predicted_cluster = 'nmf1'
omics_names = ['pro', 'rna', 'phospho']
omics_weights = [2, 1, 1]

enrichment_scores = None
cluster_names = []
for d in [pro_enrichment_scores_dir, rna_enrichment_scores_dir, phospho_enrichment_scores_dir]:
    for f in Path(d).glob('*.csv'):
        cluster = f.stem
        omics = f.parent.stem
        cluster_names.append(cluster)
        df = pd.read_csv(str(f), sep=',', index_col=0, header=0)
        if enrichment_scores is None:
            enrichment_scores = df['NES'].to_frame()
            enrichment_scores.columns = ['%s_%s' % (omics, cluster)]
        else:
            sample_names = df.index.intersection(enrichment_scores.index).tolist()
            enrichment_scores = enrichment_scores.loc[sample_names]
            enrichment_scores['%s_%s' % (omics, cluster)] = df.loc[sample_names, 'NES']

cluster_names = list(set(cluster_names))
sample_info = pd.read_csv(sample_info_file, sep=',', index_col=0, header=0)
sample_info.index = sample_info['Tumor_Sample_Barcode'].tolist()
sample_info['cluster'] = ['nmf%.1d' % cluster for cluster in sample_info['NMFcluster']]

probability = calculate_probability(
    enrichment_scores,
    cluster_names=cluster_names, predicted_cluster=predicted_cluster,
    omics_names=omics_names, omics_weights=omics_weights
)

test_preds = {}
test_labels = []
for test_index in probability.index:
    train_index = probability.index[probability.index != test_index].tolist()
    train_probability = probability.loc[train_index]
    train_y_true = (sample_info.loc[train_index, 'cluster'] == predicted_cluster).tolist()
    best_thresholds = find_best_thresholds(train_probability, train_y_true)
    test_probability = probability.loc[test_index]
    test_y_true = sample_info.loc[test_index, 'cluster'] == predicted_cluster
    for k, v in best_thresholds.items():
        if k not in test_preds:
            test_preds[k] = []
        test_preds[k].append(test_probability.loc[k].item() > best_thresholds[k])
    test_labels.append(test_y_true)
test_preds = pd.DataFrame.from_dict(test_preds, orient='columns')
metrics = calculate_metrics(test_preds, test_labels)
metrics

,accuracy,balanced_accuracy_score,precision,recall,f1
combination_pro,0.847826,0.847176,0.837209,0.837209,0.837209
combination_rna,0.750000,0.759611,0.672414,0.906977,0.772277
combination_phospho,0.673913,0.689606,0.597015,0.930233,0.727273
combination_pro_rna,0.880435,0.879212,0.880952,0.860465,0.870588
combination_pro_phospho,0.836957,0.835548,0.833333,0.813953,0.823529
combination_rna_phospho,0.750000,0.758187,0.678571,0.883721,0.767677
combination_pro_rna_phospho,0.880435,0.879212,0.880952,0.860465,0.870588


In [58]:
# nmf2 - pro: 1100, rna: 1700, phospho: 50
pro_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_1100/pro/'
rna_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_1700/rna/'
phospho_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_50/phospho/'

predicted_cluster = 'nmf2'
omics_names = ['pro', 'rna', 'phospho']
omics_weights = [1, 1, 1]

enrichment_scores = None
cluster_names = []
for d in [pro_enrichment_scores_dir, rna_enrichment_scores_dir, phospho_enrichment_scores_dir]:
    for f in Path(d).glob('*.csv'):
        cluster = f.stem
        omics = f.parent.stem
        cluster_names.append(cluster)
        df = pd.read_csv(str(f), sep=',', index_col=0, header=0)
        if enrichment_scores is None:
            enrichment_scores = df['NES'].to_frame()
            enrichment_scores.columns = ['%s_%s' % (omics, cluster)]
        else:
            sample_names = df.index.intersection(enrichment_scores.index).tolist()
            enrichment_scores = enrichment_scores.loc[sample_names]
            enrichment_scores['%s_%s' % (omics, cluster)] = df.loc[sample_names, 'NES']

cluster_names = list(set(cluster_names))
sample_info = pd.read_csv(sample_info_file, sep=',', index_col=0, header=0)
sample_info.index = sample_info['Tumor_Sample_Barcode'].tolist()
sample_info['cluster'] = ['nmf%.1d' % cluster for cluster in sample_info['NMFcluster']]

# calculate probability
probability = calculate_probability(
    enrichment_scores,
    cluster_names=cluster_names, predicted_cluster=predicted_cluster,
    omics_names=omics_names, omics_weights=omics_weights
)

# leave-one-out
test_preds = {}
test_labels = []
for test_index in probability.index:
    train_index = probability.index[probability.index != test_index].tolist()
    train_probability = probability.loc[train_index]
    train_y_true = (sample_info.loc[train_index, 'cluster'] == predicted_cluster).tolist()
    best_thresholds = find_best_thresholds(train_probability, train_y_true)
    test_probability = probability.loc[test_index]
    test_y_true = sample_info.loc[test_index, 'cluster'] == predicted_cluster
    for k, v in best_thresholds.items():
        if k not in test_preds:
            test_preds[k] = []
        test_preds[k].append(test_probability.loc[k].item() > best_thresholds[k])
    test_labels.append(test_y_true)
test_preds = pd.DataFrame.from_dict(test_preds, orient='columns')
metrics = calculate_metrics(test_preds, test_labels)
metrics

,accuracy,balanced_accuracy_score,precision,recall,f1
combination_pro,0.902174,0.821557,0.812500,0.684211,0.742857
combination_rna,0.195652,0.473684,0.197802,0.947368,0.327273
combination_phospho,0.815217,0.825162,0.533333,0.842105,0.653061
combination_pro_rna,0.880435,0.788392,0.750000,0.631579,0.685714
combination_pro_phospho,0.880435,0.807859,0.722222,0.684211,0.702703
combination_rna_phospho,0.760870,0.693583,0.440000,0.578947,0.500000
combination_pro_rna_phospho,0.847826,0.787311,0.619048,0.684211,0.650000


In [53]:
# nmf3 - pro: 100, rna: 400, phospho: 1700
pro_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_100/pro/'
rna_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_400/rna/'
phospho_enrichment_scores_dir = '/lustre/home/acct-medsjia/medsjia-wuzr/yy/prcc_multiomics/results/enrichment_scores/n_markers_1700/phospho/'

predicted_cluster = 'nmf3'
omics_names = ['pro', 'rna', 'phospho']
omics_weights = [2, 2, 1]

enrichment_scores = None
cluster_names = []
for d in [pro_enrichment_scores_dir, rna_enrichment_scores_dir, phospho_enrichment_scores_dir]:
    for f in Path(d).glob('*.csv'):
        cluster = f.stem
        omics = f.parent.stem
        cluster_names.append(cluster)
        df = pd.read_csv(str(f), sep=',', index_col=0, header=0)
        if enrichment_scores is None:
            enrichment_scores = df['NES'].to_frame()
            enrichment_scores.columns = ['%s_%s' % (omics, cluster)]
        else:
            sample_names = df.index.intersection(enrichment_scores.index).tolist()
            enrichment_scores = enrichment_scores.loc[sample_names]
            enrichment_scores['%s_%s' % (omics, cluster)] = df.loc[sample_names, 'NES']

cluster_names = list(set(cluster_names))
sample_info = pd.read_csv(sample_info_file, sep=',', index_col=0, header=0)
sample_info.index = sample_info['Tumor_Sample_Barcode'].tolist()
sample_info['cluster'] = ['nmf%.1d' % cluster for cluster in sample_info['NMFcluster']]

# calculate probability
probability = calculate_probability(
    enrichment_scores,
    cluster_names=cluster_names, predicted_cluster=predicted_cluster,
    omics_names=omics_names, omics_weights=omics_weights
)

# leave-one-out
test_preds = {}
test_labels = []
for test_index in probability.index:
    train_index = probability.index[probability.index != test_index].tolist()
    train_probability = probability.loc[train_index]
    train_y_true = (sample_info.loc[train_index, 'cluster'] == predicted_cluster).tolist()
    best_thresholds = find_best_thresholds(train_probability, train_y_true)
    test_probability = probability.loc[test_index]
    test_y_true = sample_info.loc[test_index, 'cluster'] == predicted_cluster
    for k, v in best_thresholds.items():
        if k not in test_preds:
            test_preds[k] = []
        test_preds[k].append(test_probability.loc[k].item() > best_thresholds[k])
    test_labels.append(test_y_true)
test_preds = pd.DataFrame.from_dict(test_preds, orient='columns')
metrics = calculate_metrics(test_preds, test_labels)
metrics

,accuracy,balanced_accuracy_score,precision,recall,f1
combination_pro,0.869565,0.851613,0.800000,0.800000,0.800000
combination_rna,0.847826,0.861290,0.710526,0.900000,0.794118
combination_phospho,0.315217,0.483333,0.318681,0.966667,0.479339
combination_pro_rna,0.891304,0.893548,0.794118,0.900000,0.843750
combination_pro_phospho,0.858696,0.843548,0.774194,0.800000,0.786885
combination_rna_phospho,0.836957,0.793011,0.800000,0.666667,0.727273
combination_pro_rna_phospho,0.891304,0.893548,0.794118,0.900000,0.843750
